In [ ]:
# import pandas as pd
# import numpy as np
import ast

# Movies = pd.read_csv("tmdb_5000_movies.csv")
# Credits = pd.read_csv("tmdb_5000_credits.csv")

MoviewdataFrame = Movies.merge(Credits, on="title")

contentdataFrame = MoviewdataFrame[
    ["id", "genres", "crew", "cast", "title", "keywords", "overview"]
].copy()

contentdataFrame = contentdataFrame.dropna()
contentdataFrame = contentdataFrame.drop_duplicates()


def convert(objct):
    return [i["name"] for i in ast.literal_eval(objct)]


def convert3(objct):
    return [i["name"] for i in ast.literal_eval(objct)[:3]]


def Find_Director(objct):
    try:
        for i in ast.literal_eval(objct):
            if i.get("job") == "Director":
                return [i["name"]]
    except (ValueError, SyntaxError, TypeError):
        pass
    return []


contentdataFrame["genres"] = contentdataFrame["genres"].apply(convert)
contentdataFrame["keywords"] = contentdataFrame["keywords"].apply(convert)
contentdataFrame["cast"] = contentdataFrame["cast"].apply(convert3)
contentdataFrame["director"] = contentdataFrame["crew"].apply(Find_Director)

contentdataFrame["overview"] = contentdataFrame["overview"].apply(lambda x: x.split())

contentdataFrame["director"] = contentdataFrame["director"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)
contentdataFrame["cast"] = contentdataFrame["cast"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)
contentdataFrame["genres"] = contentdataFrame["genres"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)
contentdataFrame["keywords"] = contentdataFrame["keywords"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

contentdataFrame["tags"] = (
    contentdataFrame["overview"]
    + contentdataFrame["genres"]
    + contentdataFrame["keywords"]
    + contentdataFrame["cast"]
    + contentdataFrame["director"]
)

new_df = contentdataFrame[["id", "title", "tags"]].copy()
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))

In [2]:
new_df["tags"] = new_df["tags"].apply(lambda x: x.lower())
new_df

,id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just wants to play his guitar and ...
4805,72766,Newlyweds,a newlywed couple's honeymoon is upended by th...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduces a dedic..."
4807,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...


In [3]:
new_df["tags"][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [4]:
new_df["tags"][1]

"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski"

## Need to work on the machine Learning
as this is the text based data so we cannot work on the numbers
we need to use vectotrs

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000, stop_words="english")

In [6]:
X = vectorizer.fit_transform(new_df["tags"])
X.shape

(4806, 5000)

In [7]:
vectorizer.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

In [8]:
print(type(X))

<class 'scipy.sparse._csr.csr_matrix'>


## Apply steming


In [9]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [10]:
def stemm(txt):
    L = []
    for i in txt.split():
        L.append(ps.stem(i))
    return " ".join(L)

In [11]:
stemm(
    "captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski"
)

"captain barbossa, long believ to be dead, ha come back to life and is head to the edg of the earth with will turner and elizabeth swann. but noth is quit as it seems. adventur fantasi action ocean drugabus exoticisland eastindiatradingcompani loveofone'slif traitor shipwreck strongwoman ship allianc calypso afterlif fighter pirat swashbuckl aftercreditssting johnnydepp orlandobloom keiraknightley goreverbinski"

In [12]:
new_df["tags"] = new_df["tags"].apply(stemm)

In [13]:
new_df["tags"]

0       in the 22nd century, a parapleg marin is dispa...
1       captain barbossa, long believ to be dead, ha c...
2       a cryptic messag from bond’ past send him on a...
3       follow the death of district attorney harvey d...
4       john carter is a war-weary, former militari ca...
                              ...                        
4804    el mariachi just want to play hi guitar and ca...
4805    a newlyw couple' honeymoon is upend by the arr...
4806    "signed, sealed, delivered" introduc a dedic q...
4807    when ambiti new york attorney sam is sent to s...
4808    ever sinc the second grade when he first saw h...
Name: tags, Length: 4806, dtype: str

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(X).shape

(4806, 4806)

In [17]:
similarity = cosine_similarity(X)
similarity

array([[1.        , 0.08740748, 0.05827165, ..., 0.02418254, 0.02564946,
        0.        ],
       [0.08740748, 1.        , 0.06451613, ..., 0.02677398, 0.        ,
        0.        ],
       [0.05827165, 0.06451613, 1.        , ..., 0.02677398, 0.        ,
        0.        ],
       ...,
       [0.02418254, 0.02677398, 0.02677398, ..., 1.        , 0.07071068,
        0.04836508],
       [0.02564946, 0.        , 0.        , ..., 0.07071068, 1.        ,
        0.05129892],
       [0.        , 0.        , 0.        , ..., 0.04836508, 0.05129892,
        1.        ]], shape=(4806, 4806))

In [22]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:12]

[(539, np.float64(0.25038669783359574)),
 (1192, np.float64(0.24779731389167606)),
 (507, np.float64(0.24283093212859141)),
 (260, np.float64(0.2409900932515112)),
 (1214, np.float64(0.23939494881986934)),
 (1916, np.float64(0.233785950059759)),
 (582, np.float64(0.23174488732966075)),
 (1202, np.float64(0.23084512921915967)),
 (2405, np.float64(0.23084512921915967)),
 (3728, np.float64(0.2294157338705618)),
 (1440, np.float64(0.21677749238103003))]

In [37]:
def recommend(movie):
    # 1. Look for the movie by forcing both input and titles to lowercase
    matched_movies = new_df[new_df["title"].str.lower() == movie.lower()]

    # 2. Safety Check: If no movie matches, let the user know instead of crashing
    if matched_movies.empty:
        print(f"Sorry, '{movie}' was not found in the dataset.")
        print("Try checking your spelling .")
        return

    # 3. Get the correct numerical index safely
    movie_index = matched_movies.index[0]

    # 4. Grab similarity row and sort
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[
        1:12
    ]

    # 5. Print recommendations
    print(f"Recommendations for '{new_df.iloc[movie_index].title}':\n")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)


# Test it with lowercase, spaces, or dashes
# recommend("spider-man 3")

In [38]:
recommend("troy")

Recommendations for 'Troy':

The Horseman on the Roof
Legends of the Fall
Kingdom of Heaven
The Mummy: Tomb of the Dragon Emperor
The Legend of Suriyothai
Red Cliff
Rescue Dawn
H.
Agora
Partition
The Stewardesses
